# Chapter 10 &mdash; The Nullability Rules

**Concept 6 of the Chapter 10 decomposition:** *The Nullability Rules*

A seven-case structural predicate: does $L(E)$ contain $\varepsilon$?

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Nullability-Rules/Concept-Nullability-Rules.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`nullable(E)` asks one question: **is $\varepsilon \in L(E)$?** Seven cases, all
structural:

| $E$ | nullable? |
|---|---|
| $\emptyset$ | no |
| $\varepsilon$ | **yes** |
| $a$ | no |
| $E_1+E_2$ | nullable($E_1$) **or** nullable($E_2$) |
| $E_1\&E_2$ | nullable($E_1$) **and** nullable($E_2$) |
| $E_1E_2$ | nullable($E_1$) **and** nullable($E_2$) |
| $E^*$ | **yes**, always |
| $!E$ | **not** nullable($E$) |

Two to memorise: **$E^*$ is always nullable** (zero repetitions), and
**concatenation needs *both* sides** &mdash; which is why it shares its rule with
intersection and not with union.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### The definition, checked against brute force

In [ ]:
from itertools import product
def eps_in_language(restr):
    D = min_dfa(nfa2dfa(re2nfa(restr)))
    return accepts_dfa(D, '')

## 3. Tests

The seven cases, one at a time.

In [ ]:
cases = [("''",      True,  "epsilon"),
         ("0",       False, "a single symbol"),
         ("0*",      True,  "star: zero repetitions"),
         ("0+''",    True,  "union: one side suffices"),
         ("0*1*",    True,  "concatenation: BOTH sides nullable"),
         ("0*1",     False, "concatenation: right side is not"),
         ("!(0*)",   False, "negation of a nullable expression")]
for restr, want, why in cases:
    got = nullable(re2ast(restr)[0])
    print("%-10s nullable = %-6s  %s" % (restr, got, why))
    assert got == want, restr

Against the machine route, wherever RE2NFA supports the syntax.

In [ ]:
for restr in ["''", "0", "0*", "0+1", "0*1*", "0*1", "(01)*", "(0+1)*01"]:
    assert nullable(re2ast(restr)[0]) == eps_in_language(restr), restr
print("nullable() agrees with 'the minimal DFA accepts epsilon' on every case")

**Star is always nullable**, however un-nullable its body.

In [ ]:
for body in ["0", "01", "0*1", "(0+1)(0+1)"]:
    E = re2ast("(%s)*" % body)[0]
    print("  (%s)* nullable? %s   (body nullable? %s)"
          % (body, nullable(E), nullable(re2ast(body)[0])))
    assert nullable(E)

**Concatenation needs both sides** &mdash; contrast it with union.

In [ ]:
print("%-12s %-10s %-10s" % ("E1, E2", "E1+E2", "E1.E2"))
for r1, r2 in [("0*", "1*"), ("0*", "1"), ("0", "1*"), ("0", "1")]:
    u = nullable(re2ast("(%s)+(%s)" % (r1, r2))[0])
    c_ = nullable(re2ast("(%s)(%s)" % (r1, r2))[0])
    print("%-12s %-10s %-10s" % ("%s , %s" % (r1, r2), u, c_))
assert nullable(re2ast("(0*)+(1)")[0]) and not nullable(re2ast("(0*)(1)")[0])

Nullability is what the whole algorithm rests on: it is the accept test.

In [ ]:
E = re2ast("(0+1)*01")[0]
for s in ['01', '0101', '010']:
    F = E
    for ch in s: F = dv(ch, F)
    print("  %-6r -> final expression nullable? %s" % (s, nullable(F)))
assert matches('01', E) and not matches('010', E)

## 4. Exercises


1. Write the seven rules from memory. Which two are the ones people get wrong?
2. Why do concatenation and intersection share a rule?
3. Is `nullable` ever expensive? What would make it so?

In [ ]:
# Your work for the exercises above.